# A LoRA linear layer as a custom module

Low-rank adaptation (LoRA) fine-tunes a large linear layer by freezing its weight $W$ and learning a small update $BA$ on the side:

$$y = x W^\top + b + \frac{\alpha}{r}\,(x A^\top) B^\top$$

with $W \in \mathbb{R}^{\text{out}\times\text{in}}$, $A \in \mathbb{R}^{r\times\text{in}}$, $B \in \mathbb{R}^{\text{out}\times r}$.

## The module
Write a class `LoRALinear(nn.Module)` with:
- frozen `weight` `(out, in)` and `bias` `(out,)` (stored but `requires_grad=False`),
- trainable `nn.Parameter`s `lora_A` `(r, in)` and `lora_B` `(out, r)`,
- a `forward` that accepts `x` of shape `(..., in)` (any number of leading dims) and returns `(..., out)`.

## The function that's graded
Write `lora_step(x, W, b, A, B, alpha, g)`:
- `x` `(..., in)`, `W`, `b`, `A`, `B`: float64 tensors as above; `alpha`: float; `r` is `A.shape[0]`.
- `g`: float64 tensor with the same shape as the output.

Build the module with these weights, compute `y = module(x)`, and backpropagate the scalar `(y * g).sum()`. Return a tuple

`(y, grad_A, grad_B, frozen_has_grad, n_trainable)`

- `grad_A`, `grad_B`: the gradients on the LoRA parameters.
- `frozen_has_grad`: bool, whether `weight` or `bias` ended up with a `.grad` (it should be `False`).
- `n_trainable`: int, the number of scalars in parameters with `requires_grad=True`.

This problem runs in **Colab** (PyTorch isn't available in the browser). The notebook includes the same tests.

Write your solution in the next cell, then run the test cell.

In [ ]:
import torch
from torch import nn

class LoRALinear(nn.Module):
    def __init__(self, W, b, A, B, alpha):
        super().__init__()
        self.weight = nn.Parameter(W.clone())
        self.bias = nn.Parameter(b.clone())
        self.lora_A = nn.Parameter(A.clone())
        self.lora_B = nn.Parameter(B.clone())
        self.scale = alpha

    def forward(self, x):
        return x @ self.weight.T + self.bias + self.scale * x @ self.lora_A.T @ self.lora_B.T

def lora_step(x, W, b, A, B, alpha, g):
    """Return (y, grad_A, grad_B, frozen_has_grad, n_trainable)."""
    m = LoRALinear(W, b, A, B, alpha)
    y = m(x)
    (y * g).sum().backward()
    n = sum(p.numel() for p in m.parameters())
    return y, m.lora_A.grad, m.lora_B.grad, m.weight.grad is not None, n


In [ ]:
# Tests (same harness and cases as the website)
_HARNESS = '"""Test harness shared by build.py (CPython) and the browser runner (Pyodide).\n\nOne file, so a test that passes at build time is graded the same way in the browser.\nValues cross JSON with small tags: ndarray, tuple, DataFrame/Series, non-finite floats.\n"""\nimport json, math\n\ntry:\n    import numpy as np\nexcept ImportError:  # pure-python problems never need it\n    np = None\n\n\ndef enc(x):\n    if np is not None and isinstance(x, np.ndarray):\n        return {\'__nd__\': enc(x.tolist()), \'dtype\': str(x.dtype)}\n    if np is not None and isinstance(x, np.generic):\n        return enc(x.item())\n    if isinstance(x, float) and not math.isfinite(x):\n        return {\'__f__\': repr(x)}\n    if isinstance(x, tuple):\n        return {\'__tup__\': [enc(v) for v in x]}\n    if isinstance(x, list):\n        return [enc(v) for v in x]\n    if isinstance(x, dict):\n        return {\'__dict__\': [[enc(k), enc(v)] for k, v in x.items()]}\n    t = type(x).__name__\n    if t == \'Tensor\':  # torch, in Colab and at build time\n        return enc(x.detach().cpu().numpy())\n    if t == \'DataFrame\':\n        # Cells go through enc() as Python values, so floats keep full precision (to_json caps at 15 digits).\n        return {\'__df__\': {\'columns\': [enc(c) for c in x.columns], \'data\': [enc(list(r)) for r in x.itertuples(index=False, name=None)]},\n                \'dtypes\': [str(d) for d in x.dtypes]}\n    if t == \'Series\':\n        return {\'__sr__\': enc(x.tolist()), \'name\': x.name}\n    return x\n\n\ndef dec(x):\n    if isinstance(x, list):\n        return [dec(v) for v in x]\n    if not isinstance(x, dict):\n        return x\n    if \'__nd__\' in x:\n        return np.array(dec(x[\'__nd__\']), dtype=x[\'dtype\'])\n    if \'__f__\' in x:\n        return float(x[\'__f__\'])\n    if \'__tup__\' in x:\n        return tuple(dec(v) for v in x[\'__tup__\'])\n    if \'__dict__\' in x:\n        return {_hashable(dec(k)): dec(v) for k, v in x[\'__dict__\']}\n    if \'__df__\' in x:\n        import pandas as pd\n        s = x[\'__df__\']\n        df = pd.DataFrame([dec(r) for r in s[\'data\']], columns=[dec(c) for c in s[\'columns\']])\n        for c, d in zip(df.columns, x[\'dtypes\']):\n            if d.startswith((\'int\', \'float\', \'bool\')):\n                df[c] = df[c].astype(d)\n        return df\n    if \'__sr__\' in x:\n        import pandas as pd\n        return pd.Series(dec(x[\'__sr__\']), name=x[\'name\'])\n    return x\n\n\ndef _hashable(k):\n    return tuple(k) if isinstance(k, list) else k\n\n\ndef same(got, exp, rtol=1e-6, atol=1e-8, path=\'output\'):\n    """(ok, why). Floats compare with tolerance, everything else exactly. Shapes must match."""\n    if type(exp).__name__ == \'DataFrame\':\n        if type(got).__name__ != \'DataFrame\':\n            return False, f\'{path}: expected a DataFrame, got {type(got).__name__}\'\n        if list(got.columns) != list(exp.columns):\n            return False, f\'{path}: columns {list(got.columns)} != expected {list(exp.columns)}\'\n        if got.shape != exp.shape:\n            return False, f\'{path}: shape {got.shape} != expected {exp.shape}\'\n        for c in exp.columns:\n            ok, why = same(got[c].tolist(), exp[c].tolist(), rtol, atol, f\'{path}[{c!r}]\')\n            if not ok:\n                return ok, why\n        return True, \'\'\n    if type(exp).__name__ == \'Series\':\n        exp = exp.tolist()\n        got = got.tolist() if type(got).__name__ == \'Series\' else got\n    if np is not None and isinstance(exp, np.ndarray):\n        try:\n            g = np.asarray(got)\n        except Exception:\n            return False, f\'{path}: expected an array, got {type(got).__name__}\'\n        if g.shape != exp.shape:\n            return False, f\'{path}: shape {g.shape} != expected {exp.shape}\'\n        if exp.dtype.kind in \'fc\' or g.dtype.kind in \'fc\':\n            if g.dtype.kind not in \'fciub\':\n                return False, f\'{path}: expected numbers, got dtype {g.dtype}\'\n            bad = ~np.isclose(g.astype(complex if exp.dtype.kind == \'c\' else float), exp, rtol=rtol, atol=atol, equal_nan=True)\n            if bad.any():\n                i = tuple(int(v) for v in np.argwhere(bad)[0])\n                return False, f\'{path}{list(i)}: {g[i].tolist()!r} != expected {exp[i].tolist()!r}\'\n            return True, \'\'\n        if not np.array_equal(g, exp):\n            i = tuple(int(v) for v in np.argwhere(g != exp)[0]) if g.shape else ()\n            return False, f\'{path}{list(i)}: {g[i].tolist()!r} != expected {exp[i].tolist()!r}\'\n        return True, \'\'\n    if isinstance(exp, float) or (isinstance(exp, int) and isinstance(got, float) and not isinstance(exp, bool)):\n        if np is not None and isinstance(got, np.generic):\n            got = got.item()\n        if not isinstance(got, (int, float)) or isinstance(got, bool):\n            return False, f\'{path}: expected a number, got {type(got).__name__}\'\n        if math.isnan(exp) and math.isnan(got):\n            return True, \'\'\n        return (True, \'\') if math.isclose(got, exp, rel_tol=rtol, abs_tol=atol) else (False, f\'{path}: {got!r} != expected {exp!r}\')\n    if isinstance(exp, (list, tuple)):\n        if np is not None and isinstance(got, np.ndarray):\n            got = got.tolist()\n        if not isinstance(got, (list, tuple)):\n            return False, f\'{path}: expected a {type(exp).__name__}, got {type(got).__name__}\'\n        if len(got) != len(exp):\n            return False, f\'{path}: length {len(got)} != expected {len(exp)}\'\n        for i, (g, e) in enumerate(zip(got, exp)):\n            ok, why = same(g, e, rtol, atol, f\'{path}[{i}]\')\n            if not ok:\n                return ok, why\n        return True, \'\'\n    if isinstance(exp, dict):\n        if not isinstance(got, dict):\n            return False, f\'{path}: expected a dict, got {type(got).__name__}\'\n        if set(got) != set(exp):\n            return False, f\'{path}: keys {sorted(map(str, got))} != expected {sorted(map(str, exp))}\'\n        for k in exp:\n            ok, why = same(got[k], exp[k], rtol, atol, f\'{path}[{k!r}]\')\n            if not ok:\n                return ok, why\n        return True, \'\'\n    if np is not None and isinstance(got, np.generic):\n        got = got.item()\n    return (True, \'\') if got == exp else (False, f\'{path}: {got!r} != expected {exp!r}\')\n\n\ndef run_sql(query, setup):\n    """Run `setup` (CREATE/INSERT script) in a fresh in-memory db, then `query`; rows as lists."""\n    import sqlite3  # a separate package in Pyodide: only SQL problems load it\n    con = sqlite3.connect(\':memory:\')\n    try:\n        con.executescript(setup)\n        cur = con.execute(query)\n        return [[c[0] for c in cur.description], [list(r) for r in cur.fetchall()]]\n    finally:\n        con.close()\n\n\ndef show_args(args, runtime, n=600):\n    """Arguments as the learner would write them: one per line, SQL setup as-is."""\n    if runtime == \'sql\':\n        return args[0] if len(args[0]) <= n else args[0][:n] + \' …\'\n    return \'\\n\'.join(short(a, n) for a in args)\n\n\ndef short(x, n=300):\n    s = repr(x)\n    return s if len(s) <= n else s[:n] + \' …\'\n\n\ndef grade(fn, tests, runtime, rtol=1e-6, atol=1e-8):\n    """Run `fn` (a callable, or SQL text for runtime \'sql\') on each test; list of result dicts."""\n    out = []\n    for t in tests:\n        args, exp = prep(dec(t[\'a\']), runtime), dec(t[\'e\'])\n        try:\n            got = run_sql(fn, args[0]) if runtime == \'sql\' else plain(fn(*_copy(args)))\n            ok, why = same(got, exp, rtol, atol)\n        except Exception as e:  # learner code raised: report it like a failed test\n            got, ok, why = None, False, f\'{type(e).__name__}: {e}\'\n        out.append({\'ok\': ok, \'why\': why, \'got\': short(got) if not ok else \'\',\n                    \'inp\': show_args(args, runtime, 400) if not ok else \'\'})\n    return out\n\n\ndef prep(args, runtime):\n    """Arrays cross JSON as ndarrays; torch problems take tensors."""\n    if runtime == \'torch\':\n        import torch\n        return [torch.as_tensor(a) if np is not None and isinstance(a, np.ndarray) else a for a in args]\n    if runtime == \'triton\':                                  # Colab only: kernels need CUDA tensors\n        import torch\n        return [torch.as_tensor(a).cuda() if np is not None and isinstance(a, np.ndarray) else a for a in args]\n    return args\n\n\ndef plain(x):\n    """Framework arrays (torch, jax) -> numpy, recursively, so results compare and encode the same way."""\n    t = type(x).__name__\n    if t == \'Tensor\':\n        return x.detach().cpu().numpy()\n    if t in (\'ArrayImpl\', \'Array\') and hasattr(x, \'__array__\'):\n        return np.asarray(x)\n    if isinstance(x, tuple):\n        return tuple(plain(v) for v in x)\n    if isinstance(x, list):\n        return [plain(v) for v in x]\n    if isinstance(x, dict):\n        return {k: plain(v) for k, v in x.items()}\n    return x\n\n\ndef _copy(args):\n    # Tests reuse inputs across Run and Submit, so a function that mutates its input mustn\'t poison later cases.\n    import copy\n    return copy.deepcopy(args)\n'
_TESTS = '[{"a": [{"__nd__": [[1.0, 2.0]], "dtype": "float64"}, {"__nd__": [[1.0, 0.0], [0.0, 1.0]], "dtype": "float64"}, {"__nd__": [0.0, 0.0], "dtype": "float64"}, {"__nd__": [[1.0, 1.0]], "dtype": "float64"}, {"__nd__": [[1.0], [0.0]], "dtype": "float64"}, 2.0, {"__nd__": [[1.0, 1.0]], "dtype": "float64"}], "e": {"__tup__": [{"__nd__": [[7.0, 2.0]], "dtype": "float64"}, {"__nd__": [[2.0, 4.0]], "dtype": "float64"}, {"__nd__": [[6.0], [6.0]], "dtype": "float64"}, false, 4]}}, {"a": [{"__nd__": [[1.0, -1.0, 2.0]], "dtype": "float64"}, {"__nd__": [[0.0, 0.0, 0.0], [0.0, 0.0, 0.0]], "dtype": "float64"}, {"__nd__": [0.5, -0.5], "dtype": "float64"}, {"__nd__": [[1.0, 0.0, 0.0], [0.0, 1.0, 0.0]], "dtype": "float64"}, {"__nd__": [[0.0, 0.0], [0.0, 0.0]], "dtype": "float64"}, 4.0, {"__nd__": [[1.0, 2.0]], "dtype": "float64"}], "e": {"__tup__": [{"__nd__": [[0.5, -0.5]], "dtype": "float64"}, {"__nd__": [[0.0, 0.0, 0.0], [0.0, 0.0, 0.0]], "dtype": "float64"}, {"__nd__": [[2.0, -2.0], [4.0, -4.0]], "dtype": "float64"}, false, 10]}}, {"a": [{"__nd__": [[0.35877340800391416, 1.5106773081434572, -1.7863312373111822, 1.686613508665752, -0.047317212156137566], [-0.7999785843751532, -0.802956718390362, -1.082816516582591, -0.22364535840745078, 0.8338841795521574], [0.5840637316429703, 0.6382860242561829, -1.6948083917690129, -1.5709621176914044, 1.553803174289652], [0.96888536545666, 2.18321400147264, 1.2098158348986146, -1.0243913350545566, 1.2852724548122736]], "dtype": "float64"}, {"__nd__": [[0.6283942642703312, 0.21482492767891934, -0.8198733225718833, 0.0027802218072239555, -0.14699870178042784], [0.8925351116148221, 0.12449694399871385, 1.5847459143396285, -0.7744857532181485, 0.4711025594104292], [0.49884237745245574, -0.864338539814181, -1.1812660464710163, 1.013579500325613, 0.5174404713814393]], "dtype": "float64"}, {"__nd__": [-0.1625051850123894, 0.3943673986045515, -0.6753440287843157], "dtype": "float64"}, {"__nd__": [[0.2383482020485698, -0.6051266946615304, 0.18878846665371848, 0.9860894628834065, -0.33963024501217354], [0.8497738099419226, -1.0341993407801506, -0.2884907696501737, -1.3647425964001083, -0.10058608537896181]], "dtype": "float64"}, {"__nd__": [[0.07865408332523142, -0.19726266289721334], [-0.26767571670742146, 1.3401946901039739], [1.4237928368424078, -1.2156310049465249]], "dtype": "float64"}, 8.0, {"__nd__": [[1.1741144932558947, -1.723069612870992, -0.7344792508370456], [-0.1734455959217569, 0.1110606026713016, -1.0474166760042456], [-0.7224687794067585, 0.07665381874343688, -0.6934227233827154], [-0.6412007892602988, 0.6662634522239711, 0.9317210528359195]], "dtype": "float64"}], "e": {"__tup__": [{"__nd__": [[4.423244360322849, -20.098669826046716, 19.69462907723009], [-0.7430323600699085, 2.5412051564170666, -4.575941129265955], [-1.1583245109089038, 15.487195622425746, -26.025076019700272], [-0.5883914228121143, 4.558906310686986, -14.525129545543301]], "dtype": "float64"}, {"__nd__": [[5.971834629063743, 8.823833012455001, 22.693779314031097, 0.24414702742631444, -5.998836990434596], [-4.9216630674304715, -12.846305168841349, -2.4572647694851524, -18.795084902108787, 11.350827612782991]], "dtype": "float64"}, {"__nd__": [[16.258433766045957, -20.111401195175606], [-10.684345413131734, 20.587874590532017], [-1.0482085262717749, -2.272043939921663]], "dtype": "float64"}, false, 16]}}, {"a": [{"__nd__": [[[-0.0600550310800827, 0.018197912592243334, -0.019598117164741715, -0.4544882932751738, -0.511393214091543, -1.0377445899208706], [0.6483887219942397, -0.08415967031156173, -0.13743939809669994, 0.20054261321582875, 0.5306748425469374, 1.4210345023787507], [-0.27354581958930024, -0.4011722662804173, 0.20057350347744685, -0.8455026583233285, 1.2724301176555226, 0.9262440047595556]], [[0.6939247628614134, -0.1196149568737294, 0.739582174610242, 1.0502047166386168, 0.4999059994351346, -0.2799827475865304], [-1.6850729956882953, 0.9469623538219463, -0.3532509656510693, -0.8432343818323768, 0.8905983820560427, 0.33817143735531163], [-1.4188364513624918, -0.09904137373005252, -0.2692340321763984, 0.9926118054712234, -1.895121829358109, -0.3855607500329723]]], "dtype": "float64"}, {"__nd__": [[-0.1382491179542964, 1.4189004130302252, -1.0833898591198583, -2.940431935506099, -1.3840283315854316, 0.263069762783062], [0.809592098994733, 1.7335425693272173, 0.40093512304012024, -0.8023103035286667, 0.8843895901120883, -0.011057745469750928], [-0.2009579446051938, 0.017552546089036017, 1.2745760509101105, -0.9014413693567734, -1.0239272781381537, -0.06652210021038106], [-0.06656995155514676, -0.8144586293391828, -0.8794565088883497, 0.37765916004244404, 1.337835134332041, 0.7129918729459087]], "dtype": "float64"}, {"__nd__": [0.6851211109663922, 1.2457373766175497, -1.99097272670123, -1.5103268361371722], "dtype": "float64"}, {"__nd__": [[-0.7053981939290472, -1.1951111440348927, -0.9893977243928588, 0.06987400383657381, -1.2843896578479024, -0.44349701839532474]], "dtype": "float64"}, {"__nd__": [[-2.0431546819494004], [-0.9739745699555712], [1.345547885926575], [-0.5428950996559225]], "dtype": "float64"}, 1.0, {"__nd__": [[[0.3241388671590421, 0.7081542141604098, -0.49090235523671877, 0.579084730531848], [0.21453528542881106, 0.6974051956994979, -0.6355527655967318, -1.4401713176798854], [-1.0221198928506263, -0.554816582754259, 0.5358286946534183, -0.20430898694963992]], [[-0.39880631402193534, -0.1796813723272528, 1.0997568665514252, -0.5754988975296205], [1.511331668915082, -1.227592845758537, -0.5922351807140467, 0.855002651076008], [0.5178268740164172, -0.8555092830790272, -0.10903398836430599, -0.7089297260980899]]], "dtype": "float64"}], "e": {"__tup__": [{"__nd__": [[[0.21246519099282857, 0.04862585554568288, 0.5129545990931783, -3.7105462453589784], [2.7774137853232665, 3.3414739407457024, -5.159974996810024, 1.2592533846118927], [4.235905172928336, 3.7904591063222717, -4.4832349915859755, 1.5866497350189128]], [[-1.1240662443845642, 2.982487841296595, -4.678811041362042, -0.41702295336508854], [5.913787894166828, 3.763633180788668, -3.5335984264275124, -0.2308458877001902], [-7.660048389127686, -6.605078129015832, 4.483537704207904, -5.737987065304287]]], "dtype": "float64"}, {"__nd__": [[5.6459824860938435, -4.650583579725562, 4.07123033376788, 3.4867456732003106, 3.518422506656383, 2.0773774716354407]], "dtype": "float64"}, {"__nd__": [[2.9846717405592935], [-1.3957673528683636], [-2.0177008408329935], [0.3607578948946141]], "dtype": "float64"}, false, 10]}}, {"a": [{"__nd__": [[0.8510212769423531, 0.5397406470269119, -0.7131092966031697, 0.3971833443260635]], "dtype": "float64"}, {"__nd__": [[-0.3671460370746888, -0.07191657774926997, 0.06461484502610823, -0.5640442432917747], [0.4250072828482813, 0.46028239783009683, 0.42593165582019393, 1.1046397070960905], [-0.27174114252366427, -1.4828593634501943, -0.489072005359709, 0.2007368892738041], [0.02198571377633105, 0.5786880417250706, 0.8064177974523544, -0.11758989750756722]], "dtype": "float64"}, {"__nd__": [-0.11037013953690167, -0.9874075351661735, 1.0330955204913965, -0.26393657528783593], "dtype": "float64"}, {"__nd__": [[-1.0833076865189253, -1.114373881782793, -0.4147657766285512, 1.5553992101837082], [0.7725462701194257, -1.6940422362384258, -0.12361444090580717, 0.2639596332433514], [0.6265573247505951, 0.3436628055732324, -0.6407415119480926, 0.4118864085570845], [-0.5541505601270842, -2.2406484235127633, -0.7088634775653162, -0.6859673154744348]], "dtype": "float64"}, {"__nd__": [[0.9536290088133706, 1.0560046751839265, -1.045279424641642, 0.17390955321529164], [0.09013529248460428, -1.435795498095001, -2.0531366566250346, -0.09276540124128595], [0.13644372719186973, -0.5948320751120193, -2.7111664954113537, -0.029483978631708624], [-0.5786198747278738, -0.8894063342725544, -0.32418700968801495, -1.12855669323536]], "dtype": "float64"}, 16.0, {"__nd__": [[0.5598427280686814, 0.010603864924009842, -0.5594566369571077, 0.2818726076342451]], "dtype": "float64"}], "e": {"__tup__": [{"__nd__": [[-9.9345397532408, -10.556265375595704, -14.103411395562782, 5.883751864879771]], "dtype": "float64"}, {"__nd__": [[1.0055892304635783, 0.6377718120559656, -0.84262878994526, 0.4693223358748663], [2.2400753151087875, 1.4207161826903778, -1.8770606277139625, 1.0454739843801075], [2.7860305088443793, 1.7669756917044068, -2.3345412274710555, 1.3002787884141374], [-0.6986404147079468, -0.44309659428075665, 0.5854224661702301, -0.3260650983863085]], "dtype": "float64"}, {"__nd__": [[-1.3656552237578734, -0.14309392577900235, 2.999000347373751, -3.24243128331767], [-0.025866592168578405, -0.002710312350829861, 0.05680344317468478, -0.06141421801135987], [1.3647134104290013, 0.14299524218431844, -2.9969321104927937, 3.240195166216183], [-0.6875873879400212, -0.07204569421682355, 1.5099527167682356, -1.632516624903771]], "dtype": "float64"}, false, 32]}}, {"a": [{"__nd__": [[[[-0.032100222264474815, -1.518206205158868, -0.1441467553139046], [-0.4571565407824213, -1.7407156772498007, -0.6520424779536729]], [[0.42634016461290364, -0.6194956228417741, 1.4075074257264955], [1.6982164351447544, -0.3063453638612675, 0.553434424296176]]], [[[1.9926526478415145, -0.331526522350224, -0.8989570837450346], [-0.422331995147081, -1.3742214526554224, -1.3491851888620348]], [[0.2086518433004105, -0.4167563978840294, 0.24644175655585768], [0.4883741105141886, -0.7643160931773055, -0.1944523423331713]]]], "dtype": "float64"}, {"__nd__": [[-0.2693043085863964, -0.6686649004393013, -0.8364556720740618], [0.3155505374569986, -0.14024643621485902, 0.3322867996571848], [1.0009222444961468, 1.0142174316108183, -1.1120591826841941], [-0.8558236870860543, -0.2562514323596529, 0.1236371145358894], [0.9383854586360836, -0.9621771389420167, -0.43685307558735226]], "dtype": "float64"}, {"__nd__": [0.24703569469092412, 1.2288651191177333, 1.3792342883219184, -1.0059765041187922, -0.21203724299336404], "dtype": "float64"}, {"__nd__": [[-0.8077502873624303, 0.8474791280591054, 0.4469000649128899], [-0.35779615802814096, -0.6112468227277504, 0.2949523400999831]], "dtype": "float64"}, {"__nd__": [[0.9802940600621635, -0.767444144080219], [-0.3236618111270047, 0.1129737681875106], [0.018011866501908936, 0.3378669966589955], [-0.9995766587698955, -0.5949276168950793], [-2.1001051526512886, 0.05573366268881772]], "dtype": "float64"}, 0.5, {"__nd__": [[[[1.0701569228810817, -2.057195404042072, -0.6318079969806747, 0.8985241778631176, -0.1428027365344208], [-1.8950428195675442, 1.168788331744318, -1.4259696378610343, 1.3709104657520208, -0.58464059879444]], [[-0.7065454379067378, 0.026944605818733074, -0.022175546391032475, 0.6004755228520038, 0.5025060567088484], [-0.14793754391747488, 1.3225328287068747, 0.7134748212984285, 0.27379161855350254, 0.13511514244719608]]], [[[-0.09532521278129974, -0.07758069467940808, 0.6648989933924994, 0.4742533005081589, 0.0397348380155517], [1.0151129233973155, -0.26221152794886715, 0.43589613755299295, 0.8595237833519895, -1.1155043350355576]], [[-0.2641148710183764, 1.6688064665782514, -0.9571035511645589, 1.1520796067501626, -1.0597628522710998], [0.405535390337305, -1.0641892166186686, -0.6163888649025725, -0.37531224638067917, -0.5488203633007783]]]], "dtype": "float64"}], "e": {"__tup__": [{"__nd__": [[[[0.8945745542753997, 1.5163184414512516, 0.03740987313486127, -0.40954712280983097, 1.9898248980263693], [1.5384316540016851, 1.2543795183989557, -0.0375458723853197, -0.05407268321562517, 2.0667672553990433]], [[-0.8128052905392166, 1.9555365509980438, -0.3344836088064982, -1.0733916767369347, 0.30436083043705], [-0.7582380107055835, 2.0963280159738455, 2.1249134391059026, -1.928320975771907, 2.1576008555439357]]], [[[0.27103078428537675, 1.7690098657478994, 3.961354831852191, -2.0493639778992967, 3.56223632562621], [1.9448143081123352, 0.9721846435358363, 1.1068056472122074, -0.1909604716727259, 2.060473831913034]], [[0.11400869291410703, 1.4754869767501488, 0.9108367973039689, -0.9820221319922049, 0.49671403944618187], [0.4674121911312488, 1.5235541606463212, 1.3238917466322107, -1.0049266769704803, 1.6626975207192176]]]], "dtype": "float64"}, {"__nd__": [[-0.7479388346840664, -0.213525144819686, -1.455295138749101], [0.20051770116084972, 1.095695557778658, 0.5395693388300994]], "dtype": "float64"}, {"__nd__": [[0.006459782948789426, -0.1780969155895445], [0.08064004161004641, -0.22043615533198588], [0.19792201909393214, -0.7211304842540465], [-1.4982927842779372, 0.7212247045073897], [0.8135962191286614, -0.38380572023624837]], "dtype": "float64"}, false, 16]}}, {"a": [{"__nd__": [[1.0085690589142688, 0.8334942966216093, 0.8700599664307159, 0.19348830714441678, 1.6366916490384464, 1.0287923702896826, 0.3247528649959841, -1.0487857298833259], [0.6854466141613622, 0.6521584987897013, 0.09434243617109991, 0.7128873475063768, 0.7699365242725695, -0.29845955358926546, 0.5652951110659917, 0.347065490112703], [0.3131496577718769, -0.34179643371544444, -0.46297793464524145, 0.11862428149034983, 1.4317178577246314, 2.325985422686139, -0.3698057478971567, -1.0543470939889712], [-0.34156052722203584, -0.8351430939597624, -0.4170599012110719, 0.3967821823193485, -0.9142515319015107, 0.6089667102418319, 0.7915542794734144, 1.5164360247500759], [1.8192748316879657, 0.997828907708521, 2.2671920580788893, -0.49962791416033, 0.276470890505262, 0.17205662638670258, 0.43919349351729436, 1.024591077762065], [-0.3232677308941149, 1.7133716327948385, 1.2759740433101456, 0.35307602330793086, 0.47171827985954007, -0.9634178520124883, -2.4759797049869987, -0.0809123582275483], [0.3239967842305745, 0.23537058308694503, -1.0897890047921537, -0.4421162864665036, 1.759257542559709, 0.7362004871113208, 0.6243032398185351, -0.2989905966535996]], "dtype": "float64"}, {"__nd__": [[0.7467517137104908, 1.2923755482918649, 0.7663819416213754, -0.3931861881713645, 0.29516001191068375, -0.8641129705485112, -0.2917449236407673, 0.6010780516446905], [-0.5195588624307894, -0.21733011358290186, 1.4400312683510301, -0.15393592947116394, 0.7754822078311886, -0.2849522608169767, -1.1168312675166585, 0.0183330348372163], [-0.34331274239099885, -1.983856891778598, -1.1801191577307932, -0.4033017548097564, 1.4802541598391297, -0.5561659964139166, -0.017331883732870673, 0.6397451788405071], [0.3037293059107359, -0.32103423554708943, 1.7031759481970714, 1.3443712426623422, -2.4063036979730765, 0.9680702588577331, 0.5544719540727608, 0.9869177298341354], [0.10338523921171394, 0.8337698290032067, 1.9270332857269237, 1.7130171495621231, 1.7902198650737964, 1.0600020830044186, 0.9309145546739234, 0.9497811075000052], [-1.09442440901382, 2.269770121481706, 0.2510860761421567, 0.3376584232414907, 0.13434967640702897, 0.4856635218825325, -2.088412931573151, 0.24372955871343904]], "dtype": "float64"}, {"__nd__": [-1.2410871583938738, -0.9626480973950894, -1.8295130964672903, -1.6892691709142154, 0.5960192712221675, -0.47166994919225386], "dtype": "float64"}, {"__nd__": [[-0.08914065036402882, -0.15682525475683184, -1.1393991436194493, -0.6911048845295463, -1.2390976458361571, -0.2978979313811222, -2.194688737213589, -0.3814903772748522], [0.912731508619598, 1.20625446188279, 1.1438534923955528, 0.498781081889901, -0.9924858329137045, 0.9059666097190682, 0.520598941722906, 0.2684169778476837], [-1.2004373031856796, 0.3919140032283127, 1.4058874357213444, 0.7403878069322796, -0.22429223776016513, 0.5448080755421495, 0.006254860766122802, 0.17954288578318953]], "dtype": "float64"}, {"__nd__": [[0.8423605426473773, 0.3437347798979491, 0.07454426859255363], [0.38411597350359655, -0.8126967496866394, -0.988282585949805], [0.483228178679919, -0.5877535495917325, 0.5570833818970693], [-1.827723926934063, 1.1912715587266225, -0.09833773810578231], [1.6563475651929895, -2.4594146469559424, -1.670335623109287], [1.3966502013370818, 0.15800957363487542, 0.3047770205792349]], "dtype": "float64"}, 6.0, {"__nd__": [[0.12683901406770667, 0.16268457466540617, 0.5384943889153253, -0.9986363697499262, -0.16156207823471475, 1.2970132224835127], [0.1847480249612491, -0.6114204248758205, 0.1007369315712793, 1.0435456977369353, -0.1924031722288013, -0.17070867614992088], [0.49404902758840713, 0.8514257825901366, 1.4736863204876074, -0.12171205912006818, 0.3532571718128816, 1.045571670839754], [-1.8825315671085434, 0.15230404541692436, 0.46941052441393144, -0.07183858457688504, -0.8355094739653411, 0.5943599030468357], [1.2489836306385804, -0.07383949408338283, -0.46514749878761874, -0.19825180317926416, 1.5646532222165077, 1.2544015941713493], [0.061488566602324234, -1.071458094300794, -1.238651024094895, -0.29998801234913414, -0.2925278571670005, 0.2874945668536115], [-1.1024516074091504, -0.10114598111265312, 0.5541566905155195, -1.6555993696768185, 2.3107534963187315, 0.5990995406906326]], "dtype": "float64"}], "e": {"__tup__": [{"__nd__": [[-5.08356750533958, -7.482339885350569, -9.674899606490701, 16.065596121597387, -19.017101713613812, -9.769547682839667], [-3.8041813920338017, -5.332017100976742, -6.781263503342852, 11.8776414948219, -10.86723031373184, -8.718509146204708], [-5.5937941932807735, -0.23851795473058746, -1.2594626518137741, -2.551633441963717, 3.710308950099002, -2.329239606519465], [-4.29700820998612, -6.073122366961241, -1.553227107127007, 8.186598909399516, -7.755377935428713, -5.290163719042102], [0.5392509473828491, -13.945242223063422, -15.297179001517371, 31.308190006463676, -38.73909464891892, -9.84446702553275], [9.542725520062971, 0.42225200366783566, -0.5732024257933972, -13.698181484976722, 1.279739624916884, 19.71335705188978], [-7.008928834030263, 3.244178575884075, -1.4782178709504998, -3.309656613543943, 10.810672889392292, -9.454657817167641]], "dtype": "float64"}, {"__nd__": [[32.62912856357315, 15.997731830415262, 12.595894450223815, -14.601262189562227, 49.57857711730327, 37.66415475602859, 10.17025436614918, -15.59611928515522], [-19.072229864764736, -2.1037095767541825, 8.756897569180541, 14.803646646251236, -36.74614406952817, -29.54787636442179, -17.90529484652607, 9.566861823710582], [-9.306348930954377, -2.557740244039274, -0.3293507389124704, 8.540991433127392, -10.82512316583992, -5.23656439904295, -6.670063680252281, 0.9226236234715851]], "dtype": "float64"}, {"__nd__": [[-4.686394955130203, 15.935348119816966, 5.726025992386353], [-5.11042779534638, -3.3298454210147863, -4.9811424727373605], [-14.555043320063351, -6.852553407266333, -9.553432547939375], [9.011476030804587, 1.170797954688602, 3.96259357499706], [-21.841158207095962, 6.507598846735349, -9.035519817653391], [-23.707185432047414, 17.725927667329316, 3.2766589534102613]], "dtype": "float64"}, false, 42]}}]'
import json, types
harness = types.ModuleType('harness'); exec(_HARNESS, harness.__dict__)
res = harness.grade(lora_step, json.loads(_TESTS), 'torch', *[1e-06, 1e-08])
for i, r in enumerate(res): print(f"test {i+1}: {'pass' if r['ok'] else 'FAIL  ' + r['why']}")
print(f"{sum(r['ok'] for r in res)}/{len(res)} passed")